In [6]:
!pip install kaleido==0.2.1

In [5]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os

In [7]:
results_root_dir = '/mmdetection3d/results'
plots_and_tables_dir = '/mmdetection3d/plots_and_tables'

os.makedirs(plots_and_tables_dir, exist_ok=True)

## BG proportion vs voxel size

In [ ]:
import pandas as pd
import json
import plotly.express as px

In [ ]:
with open(os.path.join(results_root_dir, 'bg_proportions.json'), 'r') as f:
    data = json.load(f)['9_sweeps']
data

In [ ]:
formated_data = []
for key, value in data.items():
    formated_data.append({
        'Voxel size': f'{key}x',
        'Maintained BG (%)': value * 100
    })

df = pd.DataFrame.from_dict(formated_data)
df

In [ ]:

fig = px.line(df, x='Voxel size', y='Maintained BG (%)', markers=True)
# Set width and height
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0), 
    width=300,
    height=250
)
fig.write_image(os.path.join(plots_and_tables_dir, "bg_proportions_plot.svg"))
fig.write_image(os.path.join(plots_and_tables_dir, "bg_proportions_plot.png"))
fig

## Points proportion per class

In [ ]:
classes_mapping = {
    'ego': {
        'classes': ['vehicle.ego'],
        'type': 'BG'
    },
    'driveable': {
        'classes': ['flat.driveable_surface'],
        'type': 'BG'
    },
    'manmade': {
        'classes': ['static.manmade'],
        'type': 'BG'
    },
    'vegetation': {
        'classes': ['static.vegetation'],
        'type': 'BG'
    },
    'sidewalk': {
        'classes': ['flat.sidewalk'],
        'type': 'BG'
    },
    'terrain': {
        'classes': ['flat.terrain'],
        'type': 'BG'
    },
    'car': {
        'classes': ['vehicle.car'],
        'type': 'FG'
    },
    'heavy': {
        'classes': ['vehicle.truck', 'vehicle.bus.bendy', 'vehicle.bus.rigid', 'vehicle.trailer', 'vehicle.construction', 'vehicle.emergency.ambulance', 'vehicle.emergency.police'],
        'type': 'FG'
    },
    'motorcycle': {
        'classes': ['vehicle.motorcycle'],
        'type': 'FG'
    },
    'bicycle': {
        'classes': ['vehicle.bicycle'],
        'type': 'FG'
    },
    'pedestrian': {
        'classes': ['human.pedestrian.adult', 'human.pedestrian.child', 'human.pedestrian.construction_worker', 'human.pedestrian.police_officer', 'human.pedestrian.wheelchair', 'human.pedestrian.stroller', 'human.pedestrian.personal_mobility'],
        'type': 'FG'
    },
    'other': {
        'classes': ['movable_object.barrier', 'movable_object.debris', 'movable_object.pushable_pullable', 'movable_object.trafficcone', 'static_object.bicycle_rack', 'flat.other', 'static.other', 'noise', 'animal'],
        'type': 'BG'
    }
}

In [ ]:

import numpy as np
from tqdm import tqdm
import os
import pandas as pd
import plotly.express as px
from mmdet3d.datasets import Seg3DDataset


def aggregate_classes_df_points_acum(df: pd.DataFrame, classes_mapping: dict) -> pd.DataFrame:
    '''
    Função de pré-processamento para aglomerar classes de pontos

    params:
        - df: DataFrame contendo a proporção de pontos. Cada linha contém uma classe, a contagem de pontos e sua proporção
        - classes_mapping: dicionário no formato:
        {
            <new_class>: {'classes': [<old_classes>], 'type': 'BG ou FG'},

        }
        Por exemplo:
        {
            'vehicle': {'classes': ['truck', 'car'], 'type': 'FG'},
            'pedestrian': {'clases': ['adult', 'child'], 'type': 'FG'}
        }
        aglomeraria as classes "truck" e "car" em apenas uma chamada "vehicle" e as classes "adult" e "child" em "pedestrian"
    
    returns:
        Um DataFrame contendo a contagem de pontos e sua proporção para as novas classes aglomeradas
    '''
    agg_classes_list = []

    for key, classes_info in classes_mapping.items():
        old_classes = classes_info['classes']
        class_type = classes_info['type']

        cur_dict = {'classes': key, 'points_total_count': 0, 'type': class_type}
        for old_class in old_classes:
            cur_dict['points_total_count'] += df[df['classes'] == old_class]['points_total_count'].values[0]
        agg_classes_list.append(cur_dict)

    df_agg = pd.DataFrame(agg_classes_list)
    df_agg = df_agg.sort_values(by='points_total_count', ascending=False)
    df_agg['percentage'] = (df_agg['points_total_count'] / df_agg['points_total_count'].sum() * 100).round(2).astype(str) + ' %'
    return df_agg

def exclude_classes_df_points_acum(df: pd.DataFrame, classes_to_exclude: 'list[str]') -> pd.DataFrame:
    '''
    Função de pré-processamento para remover classes (linhas) do DataFrame de pontos acumulados

    params:
        - df: DataFrame contendo a proporção de pontos. Cada linha contém uma classe, a contagem de pontos e sua proporção
        - classes_to_exclude: lista de classes (linhas) que devem ser excluidas
    
    returns:
        Um DataFrame contendo a contagem de pontos e sua proporção com as classes especificadas excluidas
    '''
    df_filtered = df[~df['classes'].isin(classes_to_exclude)].copy()
    df_filtered['percentage'] = (df_filtered['points_total_count'] / df_filtered['points_total_count'].sum() * 100).round(2).astype(str) + ' %'
    return df_filtered


def generate_points_acum_data(train_dataset: Seg3DDataset, val_dataset: Seg3DDataset, class_names: 'list[str]', save_path: str):
    '''
    Essa função é responsável por calculcar a quantidade TOTAL de pontos para cada classe. Caso o calculo já tenha sido feito, apenas irá carregar essa informação de um arquivo `.csv`, tornando o processo mais rápido.

    O arquivo salvo é um DataFrame contendo as colunas "classes", tendo uma linha para cada classe da base de dados (definido em `class_names`), "points_total_count" quantidade total de pontos para aquela classe, "type" BG (background) ou FG (foreground) e "percentage" a proporção de pontos daquela classe em relação ao todo.

    params:
        - train_dataset: dataset de treino para extração dos rotulos dos pontos
        - val_dataset: dataset de validação pra extração dos rotulos dos pontos
        - class_names: lista com o nome das classes dos pontos. Por exemplo, o primeiro elemento da lista será o nome dado para os pontos rotulados com o valor "0"
        - save_path: caminho onde serão salvos os dados gerados e os gráficos
    '''
    os.makedirs(save_path, exist_ok=True)
    df_path = os.path.join(save_path, 'class_proportions.csv')

    if not os.path.exists(df_path):

        points_count = np.zeros(len(class_names), dtype=np.int64)
        print('Loading from train...')
        for data in tqdm(train_dataset):
            points_count += np.bincount(data['data_samples'].gt_pts_seg.pts_semantic_mask.numpy(), minlength=len(class_names))

        print('Loading from validation...')
        for data in tqdm(val_dataset):
            points_count += np.bincount(data['data_samples'].gt_pts_seg.pts_semantic_mask.numpy(), minlength=len(class_names))

        df = pd.DataFrame(
            {
                'classes': class_names,
                'points_total_count': points_count
            }
        )
        df = df.sort_values(by='points_total_count', ascending=False)
        df['percentage'] = (df['points_total_count'] / df['points_total_count'].sum() * 100).round(2).astype(str) + ' %'

        df.to_csv(df_path, index=False)
    else:
        print('Loading saved data...')
        df = pd.read_csv(df_path)

    return df

In [ ]:
from mmengine.registry.default_scope import DefaultScope

from mmdet3d.datasets.NuScenesSegDataset import NuScenesSegDataset, classes

class_names = classes

DefaultScope.get_instance('task', scope_name='mmdet3d')  # Importante definir isso para que o escopo para a procura de módulos seja a biblioteca 
                                                         # "mmdetection3d" e não a "mmengine". Caso contrário, não encontrará os módulos de
                                                         # processamento de dados

data_root = '/mmdetection3d/data/nuscenes/'
backend_args = None
point_cloud_range = [-54.0, -54.0, -5.0, 54.0, 54.0, 3.0]

train_pipeline = [
    dict(
        type='LoadPointsFromFile',
        coord_type='LIDAR',
        load_dim=5,
        use_dim=5,
        backend_args=backend_args),
    dict(
        type='LoadAnnotations3D',  # https://github.com/open-mmlab/mmdetection3d/blob/main/mmdet3d/datasets/transforms/loading.py#L749
        with_bbox_3d=True,
        with_label_3d=True,
        with_mask_3d=False,
        with_seg_3d=True,  # New
        with_attr_label=False,
        seg_3d_dtype='np.uint8'),
    dict(
        type='Pack3DDetInputs',
        keys=[
            'points', 'pts_semantic_mask', 'gt_bboxes_3d', 'gt_labels_3d'
        ],
        meta_keys=[
            'cam2img', 'ori_cam2img', 'lidar2cam', 'lidar2img', 'cam2lidar',
            'ori_lidar2img', 'img_aug_matrix', 'box_type_3d', 'sample_idx',
            'lidar_path', 'img_path', 'num_pts_feats', 'num_views', 'lidar_sweeps', 'timestamp', 'token'
        ])
]

metainfo = dict(classes=NuScenesSegDataset.METAINFO['classes'])
lidarseg_prefix = 'lidarseg/v1.0-trainval'

data_prefix = dict(
    pts='samples/LIDAR_TOP',
    CAM_FRONT='samples/CAM_FRONT',
    CAM_FRONT_LEFT='samples/CAM_FRONT_LEFT',
    CAM_FRONT_RIGHT='samples/CAM_FRONT_RIGHT',
    CAM_BACK='samples/CAM_BACK',
    CAM_BACK_RIGHT='samples/CAM_BACK_RIGHT',
    CAM_BACK_LEFT='samples/CAM_BACK_LEFT',
    sweeps='sweeps/LIDAR_TOP',
    pts_semantic_mask=lidarseg_prefix,
    pts_instance_mask=lidarseg_prefix,
)

train_dataset = NuScenesSegDataset(
    data_root=data_root,
    ann_file='nuscenes_infos_train.pkl',
    metainfo=metainfo,
    modality=dict(use_lidar=True, use_camera=False),
    test_mode=False,
    pipeline=train_pipeline,
    data_prefix=data_prefix,
    use_valid_flag=True
)

val_dataset = NuScenesSegDataset(
    data_root=data_root,
    ann_file='nuscenes_infos_val.pkl',
    metainfo=metainfo,
    modality=dict(use_lidar=True, use_camera=False),
    test_mode=False,
    pipeline=train_pipeline,
    data_prefix=data_prefix,
    use_valid_flag=True
)

In [ ]:
df = generate_points_acum_data(
    train_dataset=train_dataset, 
    val_dataset=val_dataset, 
    class_names=class_names, 
    save_path=results_root_dir
)
df = exclude_classes_df_points_acum(aggregate_classes_df_points_acum(df, classes_mapping), ['ego'])


In [ ]:
fig = px.bar(
    df,
    x='points_total_count',
    y='classes',
    color='type',
    text='percentage',
    orientation='h',
    category_orders={"classes": df["classes"].tolist()},
    color_discrete_map={
        "FG": "#9bbbff",
        "BG": "#ff8a8a"
    }
)

threshold = df["points_total_count"].max() * 0.2

for trace in fig.data:
    positions = [
        "outside" if value < threshold else "inside"
        for value in trace.x
    ]

    trace.textposition = positions

new_names = {
    "FG": "FG (8%)",
    "BG": "BG (92%)"
}

for trace in fig.data:
    if trace.name in new_names:
        trace.name = new_names[trace.name]

fig.update_layout(
    legend=dict(
        title_text="",
        x=0.55,
        y=0.2,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.5)"
    ),
    xaxis=dict(
        title=None,
        showticklabels=True,
        showgrid=True
    ),
    yaxis=dict(
        title=None,
        showgrid=False,
        zeroline=False
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    width=300,
    height=300
)

fig.write_image(os.path.join(plots_and_tables_dir, "bg_fg_proportions.svg"))
fig.write_image(os.path.join(plots_and_tables_dir, "bg_fg_proportions.png"))

fig

## Boxplotes

In [ ]:
import plotly.graph_objects as go
import os
from collections.abc import Callable
import pandas as pd
import json
import plotly.express as px

def generate_points_descriptive_data(train_dataset: Seg3DDataset, val_dataset: Seg3DDataset, class_names: 'list[str]', save_path: str, extra_data_getters: dict={}):
    '''
    Essa função gera um DataFrame o qual cada linha é uma amostra. Existem colunas com os nomes de cada classe de pontos em `class_names`. Essas colunas com os nomes das classes possuem a contagem de pontos pertencente àquela classe naquela amostra. Colunas extras podem ser criadas usando o parâmetro `extra_data_getters`.

    params:
        - train_dataset: dataset de treino para extração dos rotulos dos pontos
        - val_dataset: dataset de validação pra extração dos rotulos dos pontos
        - class_names: lista com o nome das classes dos pontos. Por exemplo, o primeiro elemento da lista será o nome dado para os pontos rotulados com o valor "0"
        - save_path: caminho onde serão salvos os dados gerados e os gráficos
        - extra_data_getters: parâmetro responsável por criar os dados para novas colunas. É um dicionário seguindo o padrão abaixo:
        {
            <nome_nova_coluna>: <funcao_recebendo_um_registro_do_dataset_e_que_retorna_uma_informação_extra_para_adicionar_no_dataframe_gerado>
        }
    '''
    question_2_df_path = os.path.join(save_path, 'descriptive_data.csv')

    if not os.path.exists(question_2_df_path):
        points_count = np.zeros((len(train_dataset) + len(val_dataset), len(class_names)), dtype=np.int64)
        extra_data = {key: [] for key in extra_data_getters.keys()}
        print('Loading from train...')
        for i, data in enumerate(tqdm(train_dataset)):
            points_count[i] = np.bincount(data['data_samples'].gt_pts_seg.pts_semantic_mask.numpy(), minlength=len(class_names))
            for key, get_extra_data in extra_data_getters.items():
                extra_data[key].append(get_extra_data(data))

        print('Loading from validation...')
        for i, data in enumerate(tqdm(val_dataset)):
            points_count[i+len(train_dataset)] = np.bincount(data['data_samples'].gt_pts_seg.pts_semantic_mask.numpy(), minlength=len(class_names))
            for key, get_extra_data in extra_data_getters.items():
                extra_data[key].append(get_extra_data(data))

        df_pts_question_2 = pd.DataFrame(
            {
                **extra_data,
                **{class_name: points_count[:, i] for i, class_name in enumerate(class_names)}
            }
        )

        df_pts_question_2.to_csv(question_2_df_path, index=False)
    else:
        print('Loading saved data...')
        df_pts_question_2 = pd.read_csv(question_2_df_path)

    return df_pts_question_2

def generate_proportions_df_points_descriptive(df: pd.DataFrame, classes_names: 'list[str]') -> pd.DataFrame:
    '''
    Gera um DataFrame contendo a proporção (ao invés da contagem) de pontos para cada classe em cada linha do DataFrame fornecido.

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são uma contagem dos pontos daquela classe
        - classes_names: lista de classes (colunas) que possuem a contagem de pontos
    
    returns:
        Um novo DataFrame com as mesmas colunas, porém, ao invés de conter a contagem dos pontos, agora é a proporção (de 0 a 1) de pontos daquela classe em cada registro.
    '''
    df_proportion = df.copy()
    df_proportion[classes_names] = df_proportion[classes_names] / df_proportion[classes_names].sum(axis=1).to_numpy().reshape(-1, 1)
    return df_proportion

def generate_bgfg_columns_df_points_descriptive(df: pd.DataFrame, class_names: 'list[str]', is_bg_point_class_name: 'Callable[[str], bool]') -> pd.DataFrame:
    '''
    Gera um DataFrame contendo apenas a contagem em termos das classes de background (BG) e foreground (FG)

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são uma contagem dos pontos daquela classe
        - classes_names: lista de classes (colunas) que possuem a contagem de pontos
        - is_bg_point_class_name: uma função que recebe como entrada um valor contido em `classes_names` e retorna True se for uma classe de BG e False caso seja de FG.
    
    returns:
        Um novo DataFrame com as contagems (ou proporções) de BG e FG (colunas nomeadas por 'BG' e 'FG')
    '''
    df_bgfg = df.copy()

    df_bgfg['BG'] = 0
    df_bgfg['FG'] = 0

    for class_name in class_names:
        if is_bg_point_class_name(class_name):
            df_bgfg['BG'] += df_bgfg[class_name]
        else:
            df_bgfg['FG'] += df_bgfg[class_name]
    
    df_bgfg = df_bgfg[[col for col in df.columns if col not in class_names] + ['BG', 'FG']]

    return df_bgfg








def aggregate_classes_df_points_descriptive(df: pd.DataFrame, classes_mapping: dict) -> pd.DataFrame:
    '''
    Função de pré-processamento para aglomerar classes de pontos

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são uma contagem dos pontos daquela classe
        - classes_mapping: dicionário no formato:
        {
            <new_class>: {'classes': [<old_classes>], 'type': 'BG ou FG'},

        }
        Por exemplo:
        {
            'vehicle': {'classes': ['truck', 'car'], 'type': 'FG'},
            'pedestrian': {'clases': ['adult', 'child'], 'type': 'FG'}
        }
        aglomeraria as classes "truck" e "car" em apenas uma chamada "vehicle" e as classes "adult" e "child" em "pedestrian"
    
    returns:
        Um DataFrame contendo a contagem de pontos para as novas classes aglomeradas
    '''
    df_agg = df.copy()

    all_old_clasess = []

    for key, classes_info in classes_mapping.items():
        df_agg[key] = df_agg[classes_info['classes']].sum(axis=1)
        all_old_clasess.extend(classes_info['classes'])

    df_agg = df_agg.drop(columns=all_old_clasess)
    return df_agg

def group_by_df_points_descriptive(df: pd.DataFrame, class_names: 'list[str]', group_by_col: str) -> pd.DataFrame:
    '''
    Gera um DataFrame agrupado pelo parâmetro `group_by_col`. As outras colunas terão seus valores somados (acumulados).

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são uma contagem dos pontos daquela classe
        - classes_names: lista de classes (colunas) que possuem a contagem de pontos
        - group_by_col: nome da coluna que será utilizada para o agrupamento
    
    returns:
        Um novo DataFrame com o resultado do agrupamento
    '''
    df_grouped = df[[group_by_col, *class_names]]
    df_grouped = df_grouped.groupby(group_by_col).sum().reset_index()
    return df_grouped

def group_by_df_points_descriptive_mean(df: pd.DataFrame, class_names: 'list[str]', group_by_col: str) -> pd.DataFrame:
    '''
    Gera um DataFrame agrupado pelo parâmetro `group_by_col`. As outras colunas terão seus valores somados (acumulados).

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são uma contagem dos pontos daquela classe
        - classes_names: lista de classes (colunas) que possuem a contagem de pontos
        - group_by_col: nome da coluna que será utilizada para o agrupamento
    
    returns:
        Um novo DataFrame com o resultado do agrupamento
    '''
    df_grouped = df[[group_by_col, *class_names]]
    df_grouped = df_grouped.groupby(group_by_col).mean().reset_index()
    return df_grouped


def exclude_classes_df_points_descriptive(df: pd.DataFrame, classes_to_exclude: 'list[str]') -> pd.DataFrame:
    '''
    Função de pré-processamento para remover classes (colunas) do DataFrame de pontos acumulados

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são uma contagem dos pontos daquela classe
        - classes_to_exclude: lista de classes (colunas) que devem ser excluidas
    
    returns:
        Um DataFrame contendo a contagem de pontos e sua proporção com as classes especificadas excluidas
    '''
    return df.drop(columns=classes_to_exclude)







def save_points_descriptive_table(df: pd.DataFrame, classes_names: 'list[str]', save_path: str, file_posfix: str = ''):
    '''
    Gera uma tabela com dados descritivos (média, mediana, Q1, Q3, etc.) das colunas `classes_names` do DataFrame `df`.

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são a proporção daquela classe no registro
        - classes_names: lista de classes (colunas) que possuem a proporção de pontos
        - save_path: caminho que será salvo a tabela (`<save_path>/df_desc<save_pos_fix>.csv` e '.md')
        - save_pos_fix: pós-fixo usado no caminho de salvamento da tabela (`<save_path>/df_desc<save_pos_fix>.csv` e '.md')
    '''
    df_desc = df[classes_names].describe()
    df_desc = df_desc.round(2)

    df_desc.to_csv(os.path.join(save_path, f'df_desc{file_posfix}.csv'))
    df_desc.to_markdown(os.path.join(save_path, f'df_desc{file_posfix}.md'))

def plot_and_save_points_descriptive_box_plot(df: pd.DataFrame, cols: 'list[str]', save_path: str):
    '''
    Gera boxes plotes tabela com os dados das colunas `classes_names` do DataFrame `df`.

    params:
        - df: DataFrame contendo a proporção de pontos. Cada coluna é uma classe e os valores são a proporção daquela classe no registro
        - cols: lista de colunas que serão usadas no plot
        - save_path: caminho que será salvo o gráfico (`<save_path>/box_plot.png` e '.html')
    '''
    fig = go.Figure()
    for col in cols:
        fig.add_trace(go.Box(x=df[col], name=col))

    fig.show()

    fig.write_html(os.path.join(save_path, 'box_plot.html'))
    fig.write_image(os.path.join(save_path, 'box_plot.png'))


def generate_points_descriptive_df(df, is_bg_point_class_name: 'Callable[[str], bool]', preprocess_func: 'Callable[[pd.DataFrame], pd.DataFrame]'=lambda df: df, extra_data_getters: dict = {}):
    preprocessed_df = preprocess_func(df)

    pos_preprocess_classes = [col for col in preprocessed_df.columns if col not in list(extra_data_getters.keys())]

    preprocessed_df_bgfg = generate_bgfg_columns_df_points_descriptive(preprocessed_df, pos_preprocess_classes, is_bg_point_class_name)
    preprocessed_df_bgfg_proportion = generate_proportions_df_points_descriptive(preprocessed_df_bgfg, ['BG', 'FG'])

    return preprocessed_df_bgfg_proportion

def is_bg_point_class_name(class_name: str) -> bool:
    return classes_mapping[class_name]['type'] == 'BG'

In [ ]:
from nuscenes.nuscenes import NuScenes
nusc = NuScenes(version='v1.0-trainval', dataroot='/mmdetection3d/data/nuscenes', verbose=True)

extra_data_getters = {
    'sample_token': lambda data: data['data_samples'].token,
    'scene_token': lambda data: nusc.get('sample', data['data_samples'].token)['scene_token']
}

In [ ]:
df_samples = generate_points_descriptive_data(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    class_names=class_names,
    save_path=results_root_dir,
    extra_data_getters=extra_data_getters
)

df_samples = generate_points_descriptive_df(
    df_samples,
    is_bg_point_class_name=is_bg_point_class_name,
    preprocess_func=lambda df: exclude_classes_df_points_descriptive(
        aggregate_classes_df_points_descriptive(
            df, classes_mapping
        ), ['ego']
    ),
    extra_data_getters=extra_data_getters
)

In [ ]:
df_scenes = generate_points_descriptive_data(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    class_names=class_names,
    save_path=results_root_dir,
    extra_data_getters=extra_data_getters
)

agg_classes_names = list(classes_mapping.keys())

df_scenes = generate_points_descriptive_df(
    df_scenes,
    is_bg_point_class_name=is_bg_point_class_name,
    preprocess_func=lambda df: exclude_classes_df_points_descriptive(
        group_by_df_points_descriptive(
            aggregate_classes_df_points_descriptive(
                df, classes_mapping
            ), 
            agg_classes_names, 'scene_token'
        ), ['ego']
    ),
    extra_data_getters=extra_data_getters
)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Box(x=df_samples['BG'], name='Sample'))
fig.add_trace(go.Box(x=df_scenes['BG'], name='Scene'))

fig.update_layout(
    showlegend=False,
    margin=dict(l=0, r=0, t=0, b=0), 
    width=300,
    height=100,
)

fig.write_image(os.path.join(plots_and_tables_dir, "bg_proportions_box_plot.png"))
fig.write_image(os.path.join(plots_and_tables_dir, "bg_proportions_box_plot.svg"))

fig

In [ ]:
df_samples

### Outliers analysis

In [ ]:
Q1 = df_samples['BG'].quantile(0.25)
Q3 = df_samples['BG'].quantile(0.75)
IQR = Q3 - Q1

df_outliers = df_samples[(df_samples['BG'] < (Q1 - 1.5 * IQR)) | (df_samples['BG'] > (Q3 + 1.5 * IQR))]
df_outliers = df_outliers.sort_values(by='BG', ascending=False)
df_outliers = df_outliers.reset_index(drop=True)
df_outliers['label'] = 'not_labeled'

if not os.path.exists(os.path.join(results_root_dir, 'outliers.csv')):
    df_outliers.to_csv(os.path.join(results_root_dir, 'outliers.csv'), index=False)

In [ ]:
df_outliers.to_csv(os.path.join(results_root_dir, 'outliers.csv'), index=False)

from typing import List
import matplotlib.pyplot as plt
from nuscenes.utils.geometry_utils import BoxVisibility

def render_sample(nusc: NuScenes,
    token: str,
    box_vis_level: BoxVisibility = BoxVisibility.ANY,
    nsweeps: int = 1,
    out_path: str = None,
    show_lidarseg: bool = False,
    filter_lidarseg_labels: List = None,
    lidarseg_preds_bin_path: str = None,
    verbose: bool = True,
    show_panoptic: bool = False
) -> None:
        """
        Render all LIDAR and camera sample_data in sample along with annotations.
        :param token: Sample token.
        :param box_vis_level: If sample_data is an image, this sets required visibility for boxes.
        :param nsweeps: Number of sweeps for lidar and radar.
        :param out_path: Optional path to save the rendered figure to disk.
        :param show_lidarseg: Whether to show lidar segmentations labels or not.
        :param filter_lidarseg_labels: Only show lidar points which belong to the given list of classes.
        :param lidarseg_preds_bin_path: A path to the .bin file which contains the user's lidar segmentation
                                        predictions for the sample.
        :param verbose: Whether to show the rendered sample in a window or not.
        :param show_panoptic: When set to True, the lidar data is colored with the panoptic labels. When set
            to False, the colors of the lidar data represent the distance from the center of the ego vehicle.
            If show_lidarseg is True, show_panoptic will be set to False.
        """
        record = nusc.get('sample', token)

        # Separate RADAR from LIDAR and vision.
        camera_data = {}
        for channel, token in record['data'].items():
            sd_record = nusc.get('sample_data', token)
            sensor_modality = sd_record['sensor_modality']

            if sensor_modality == 'camera':
                camera_data[channel] = token

        fig, axes = plt.subplots(2, 3, figsize=(24, 10))
        
        axes_map = {
            'CAM_FRONT_LEFT': axes[0, 0],
            'CAM_FRONT': axes[0, 1],
            'CAM_FRONT_RIGHT': axes[0, 2],
            'CAM_BACK_RIGHT': axes[1, 0],
            'CAM_BACK': axes[1, 1],
            'CAM_BACK_LEFT': axes[1, 2]
        }

        # Plot cameras in separate subplots.
        for _, sd_token in camera_data.items():
            if show_lidarseg or show_panoptic:
                sd_record = nusc.get('sample_data', sd_token)
                sensor_channel = sd_record['channel']
                valid_channels = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
                                  'CAM_BACK_LEFT', 'CAM_BACK', 'CAM_BACK_RIGHT']
                assert sensor_channel in valid_channels, 'Input camera channel {} not valid.'.format(sensor_channel)
                ax = axes_map[sensor_channel]

                nusc.explorer.render_pointcloud_in_image(record['token'],
                                                pointsensor_channel='LIDAR_TOP',
                                                camera_channel=sensor_channel,
                                                show_lidarseg=show_lidarseg,
                                                filter_lidarseg_labels=filter_lidarseg_labels,
                                                ax=ax, verbose=False,
                                                lidarseg_preds_bin_path=lidarseg_preds_bin_path,
                                                show_panoptic=show_panoptic)
            else:
                nusc.explorer.render_sample_data(sd_token, box_vis_level=box_vis_level, ax=ax, nsweeps=nsweeps,
                                        show_lidarseg=False, verbose=False)

        # Change plot settings and write to disk.
        axes.flatten()[-1].axis('off')
        plt.tight_layout()
        fig.subplots_adjust(wspace=0, hspace=0)

        if out_path is not None:
            plt.savefig(out_path)

        if verbose:
            plt.show()
        
        plt.close()

from tqdm import tqdm
import os

imgs_savepath = os.path.join(results_root_dir, 'outliers_imgs')
os.makedirs(imgs_savepath, exist_ok=True)

for sample_token in tqdm(df_outliers['sample_token']):
    img_final_path = os.path.join(imgs_savepath, f'{sample_token}.png')
    if os.path.exists(img_final_path):
        continue
    render_sample(
        nusc, 
        sample_token, 
        out_path=img_final_path,
        verbose=False,
        show_lidarseg=True
    )

#### Labeling code

You can use the code below to label yourself the data

In [ ]:
!pip install ipywidgets

In [ ]:
import ipywidgets as widgets
import pandas as pd
import os

In [ ]:
df_outliers = pd.read_csv(os.path.join(results_root_dir, 'outliers.csv'))

In [ ]:
img_widget = widgets.Image(
    format='png',
    width=2400,
    height=1000
)

remaining_outliers_label = widgets.Label(value="", layout=widgets.Layout(margin='0 20px 0 0', font_size='20px'))

big_vehicle_near_btn = widgets.Button(description="Big Vehicle Near", button_style='primary', layout=widgets.Layout(margin='0 20px 0 0'), font_size='14px')
busy_btn = widgets.Button(description="Busy", button_style='primary', layout=widgets.Layout(margin='0 20px 0 0'), font_size='14px')
other_btn = widgets.Button(description="Other", button_style='primary', layout=widgets.Layout(margin='0 20px 0 0'), font_size='14px')

In [ ]:

sample = None

def load_random_unlabeled_sample(df):
    global sample
    sample = df[df['label'] == 'not_labeled'].sample(n=1).iloc[0]
    img_widget.value = open(os.path.join(results_root_dir, 'outliers_imgs', f'{sample["sample_token"]}.png'), 'rb').read()

def update_outliers_csv(df, label):
    df.loc[df['sample_token'] == sample['sample_token'], 'label'] = label
    df.to_csv(os.path.join(results_root_dir, 'outliers.csv'), index=False)

def update_remaining_outliers_label(df):
    remaining = len(df[df['label'] != 'not_labeled'])
    remaining_outliers_label.value = f"{remaining}/{len(df)} ({(remaining/len(df))*100:.2f}%)"

def on_label_btn_clicked(label):    
    update_outliers_csv(df, label)
    load_random_unlabeled_sample(df)
    update_remaining_outliers_label(df)

In [ ]:
load_random_unlabeled_sample(df_outliers)
update_remaining_outliers_label(df_outliers)

In [ ]:
big_vehicle_near_btn.on_click(lambda _: on_label_btn_clicked('big_vehicle_near'))
busy_btn.on_click(lambda _: on_label_btn_clicked('busy'))
other_btn.on_click(lambda _: on_label_btn_clicked('other'))

In [ ]:
display(img_widget)
display(widgets.HBox([remaining_outliers_label, other_btn, busy_btn, big_vehicle_near_btn], layout=widgets.Layout(justify_content='flex-end')))

#### Use labeled data

Or you can download the labeled data

In [26]:
!pip install gdown

In [29]:
import gdown

file_id = "1WBY1JgDSmZU68CNUBQZdxOZhbltGyCh3"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, os.path.join(results_root_dir, 'outliers.csv'), quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1WBY1JgDSmZU68CNUBQZdxOZhbltGyCh3
To: /mmdetection3d/results/outliers.csv
100%|██████████| 213k/213k [00:00<00:00, 3.87MB/s]


'/mmdetection3d/results/outliers.csv'

#### Final stats and outlier samples

In [31]:
df_outliers = pd.read_csv(os.path.join(results_root_dir, 'outliers.csv'))
df_outliers

,sample_token,scene_token,BG,FG,label
0,00889f8a9549450aa2f32cf310a3e305,2fc3753772e241f2ab2cd16a784cc680,0.726688,0.273312,big_vehicle_near
1,a0e65e293c764ccb9aeb8e6f446db631,daa9fce50073470398da3c2bccdaf21b,0.726687,0.273313,big_vehicle_near
2,5f9dc902d6d84566b077a1bc3721feac,f55c8c62235146d895f611176fd0a5dc,0.726660,0.273340,big_vehicle_near
3,9ff1ef84f21a4a379acfed94c33be35e,3b5f3391078e48ac92600cbdfa4ccbe6,0.726607,0.273393,big_vehicle_near
4,d87a8d65103246918d76b491cce59773,84e056bd8e994362a37cba45c0f75558,0.726538,0.273462,big_vehicle_near
...,...,...,...,...,...
1783,b37cc7e34e5345f9b21ac9e9807909e1,0d2cc345342a460e94ff54748338ac22,0.421499,0.578501,busy
1784,4e5daeb62e474bd98b7938739bc022ff,d95a36c034344ae2ac11daf2ba1b2c60,0.409171,0.590829,big_vehicle_near
1785,255d5c5c0c3e4d7387ce4a7e9812cb82,b519ec833e23450a8bd3340b67f2516b,0.404307,0.595693,big_vehicle_near
1786,7add2579384f4fe4b345dc47e1523b11,bed8426a524d45afab05b19cf02386b2,0.398606,0.601394,big_vehicle_near


In [32]:
df_outliers['label'].value_counts()

label
big_vehicle_near    1427
busy                 361
Name: count, dtype: int64

In [33]:
df_outliers['label'].value_counts() / len(df_outliers)

label
big_vehicle_near    0.798098
busy                0.201902
Name: count, dtype: float64

## Results latex tables

In [16]:
import pandas as pd
import numpy as np
import os

In [17]:
df = pd.read_csv(os.path.join(results_root_dir, 'evaluation_results.csv'))
df

,model_name,model_category,trained_on_filter,dataset,filter_type,mAP,NDS,flops,fps_1,fps_5,peak_memory_allocated_gb,peak_memory_reserved_gb,co2_kg
0,bevfusion_lidar_3dh,voxel,False,nuscenes,no_filter,0.638657,0.691101,2.304831e+11,17.216770,32.905030,2.802252,4.031250,0.003098
1,bevfusion_lidar_3dh,voxel,True,nuscenes,1x,0.666361,0.701666,1.136462e+11,22.055646,65.882156,2.782773,3.771484,0.001581
2,bevfusion_lidar_3dh,voxel,False,nuscenes,1x,0.624233,0.679719,1.136452e+11,22.002129,65.935155,2.784235,3.751953,0.001686
3,bevfusion_lidar_3dh,voxel,False,nuscenes,2x,0.622159,0.678945,1.137938e+11,21.987146,65.865696,2.783653,3.783203,0.001597
4,bevfusion_lidar_3dh,voxel,False,nuscenes,4x,0.626127,0.681632,1.144872e+11,21.967718,65.736583,2.783583,3.751953,0.001644
5,bevfusion_lidar_3dh,voxel,False,nuscenes,8x,0.635066,0.686545,1.156407e+11,21.960243,65.069696,2.786489,3.783203,0.001651
6,bevfusion_lidar_3dh,voxel,False,nuscenes,16x,0.639469,0.688805,1.183244e+11,21.808882,64.841081,2.788597,3.751953,0.001674
7,bevfusion_lidar_3dh,voxel,False,nuscenes,32x,0.642577,0.691745,1.226794e+11,21.163964,63.556858,2.788537,3.787109,0.001747
8,bevfusion_lidar,voxel,False,nuscenes,no_filter,0.642859,0.691438,2.460861e+11,21.781513,38.617019,2.805007,4.046875,0.002514
9,bevfusion_lidar,voxel,True,nuscenes,1x,0.671862,0.704306,1.856696e+11,26.757560,60.982850,2.785443,3.802734,0.001694


In [18]:
df_latex = df[(df['filter_type'] == 'no_filter') | ((df['filter_type'] == '1x') & (df['trained_on_filter'] == True))].copy()
df_latex['GFLOPS'] = df_latex['flops'] / 10 ** 9
df_latex['co2_g'] = df_latex['co2_kg'] * 1000
models_order = [
    'bevfusion_lidar',
    'bevfusion_lidar_3dh',
    'centerpoint_voxel',
    'centerpoint_voxel_3dh',
    'centerpoint_pillar',
    'ssn',
    'pointpillars'
]
df_latex = df_latex.set_index('model_name').loc[models_order].reset_index()

models_name_map = {
    'bevfusion_lidar': 'BEVFusion-L',
    'bevfusion_lidar_3dh': 'BEVFusion-L 3Dh',
    'centerpoint_voxel': 'CenterPoint-Voxel',
    'centerpoint_voxel_3dh': 'CenterPoint-Voxel 3Dh',
    'centerpoint_pillar': 'CenterPoint-Pillar',
    'ssn': 'SSN',
    'pointpillars': 'PointPillars'
}
df_latex['model_name'] = df_latex['model_name'].map(models_name_map)

df_latex

,model_name,model_category,trained_on_filter,dataset,filter_type,mAP,NDS,flops,fps_1,fps_5,peak_memory_allocated_gb,peak_memory_reserved_gb,co2_kg,GFLOPS,co2_g
0,BEVFusion-L,voxel,False,nuscenes,no_filter,0.642859,0.691438,2.460861e+11,21.781513,38.617019,2.805007,4.046875,0.002514,246.086098,2.514141
1,BEVFusion-L,voxel,True,nuscenes,1x,0.671862,0.704306,1.856696e+11,26.757560,60.982850,2.785443,3.802734,0.001694,185.669640,1.694187
2,BEVFusion-L 3Dh,voxel,False,nuscenes,no_filter,0.638657,0.691101,2.304831e+11,17.216770,32.905030,2.802252,4.031250,0.003098,230.483098,3.098027
3,BEVFusion-L 3Dh,voxel,True,nuscenes,1x,0.666361,0.701666,1.136462e+11,22.055646,65.882156,2.782773,3.771484,0.001581,113.646160,1.581033
4,CenterPoint-Voxel,voxel,False,nuscenes,no_filter,0.556577,0.642017,1.635292e+11,14.549825,26.434894,1.061831,1.583984,0.003272,163.529176,3.271976
5,CenterPoint-Voxel,voxel,True,nuscenes,1x,0.611984,0.668890,1.216225e+11,17.176213,39.008053,0.452675,1.121094,0.001542,121.622519,1.542071
6,CenterPoint-Voxel 3Dh,voxel,False,nuscenes,no_filter,0.567951,0.650441,1.673027e+11,5.939207,7.873990,1.061503,1.585938,0.005712,167.302681,5.712041
7,CenterPoint-Voxel 3Dh,voxel,True,nuscenes,1x,0.605197,0.663655,8.626479e+10,13.332940,32.354555,0.451675,1.130859,0.001489,86.264785,1.489413
8,CenterPoint-Pillar,pillar,False,nuscenes,no_filter,0.482006,0.593399,1.278589e+11,21.981053,48.945499,3.217486,20.984375,0.002592,127.858931,2.591720
9,CenterPoint-Pillar,pillar,True,nuscenes,1x,0.576202,0.645340,1.272515e+11,34.484618,71.062425,0.822111,1.695312,0.001399,127.251504,1.399492


In [19]:
df_result = df_latex.copy()

model_names = df_latex['model_name'].unique()

cols_round_1 = ['fps_1', 'fps_5']
cols_round_2 = ['GFLOPS', 'peak_memory_allocated_gb', 'co2_g']
cols_round_3 = ['mAP', 'NDS']

for model_name in model_names:

    mask_ori = (
        (df_latex['model_name'] == model_name) &
        (~df_latex['trained_on_filter'])
    )

    mask_filter = (
        (df_latex['model_name'] == model_name) &
        (df_latex['trained_on_filter'])
    )

    df_model_ori = df_latex.loc[mask_ori]
    df_model_filter = df_latex.loc[mask_filter]
    
    enhancement_1 = (
        (df_model_filter[cols_round_1].to_numpy(dtype=float)
         - df_model_ori[cols_round_1].to_numpy(dtype=float))
        / df_model_ori[cols_round_1].to_numpy(dtype=float)
        * 100
    )

    enhancement_2 = (
        (df_model_filter[cols_round_2].to_numpy(dtype=float)
         - df_model_ori[cols_round_2].to_numpy(dtype=float))
        / df_model_ori[cols_round_2].to_numpy(dtype=float)
        * 100
    )

    enhancement_3 = (
        (df_model_filter[cols_round_3].to_numpy(dtype=float)
         - df_model_ori[cols_round_3].to_numpy(dtype=float))
        / df_model_ori[cols_round_3].to_numpy(dtype=float)
        * 100
    )

    enhancement_1 = np.round(enhancement_1, 1)
    enhancement_2 = np.round(enhancement_2, 1)
    enhancement_3 = np.round(enhancement_3, 1)

    enh_1_str = np.char.mod('%.1f', enhancement_1)
    enh_1_str = np.where(enhancement_1 > 0, np.char.add('+', enh_1_str), enh_1_str)

    enh_2_str = np.char.mod('%.1f', enhancement_2)
    enh_2_str = np.where(enhancement_2 > 0, np.char.add('+', enh_2_str), enh_2_str)

    enh_3_str = np.char.mod('%.1f', enhancement_3)
    enh_3_str = np.where(enhancement_3 > 0, np.char.add('+', enh_3_str), enh_3_str)

    ori_1_str = df_model_ori[cols_round_1].round(1).astype(str).values
    ori_2_str = df_model_ori[cols_round_2].round(2).astype(str).values
    ori_3_str = df_model_ori[cols_round_3].round(3).astype(str).values

    filt_1_str = df_model_filter[cols_round_1].round(1).astype(str).values
    filt_2_str = df_model_filter[cols_round_2].round(2).astype(str).values
    filt_3_str = df_model_filter[cols_round_3].round(3).astype(str).values

    df_result.loc[mask_filter, cols_round_1] = (
        filt_1_str + " (" + enh_1_str + "\\%)"
    )
    
    df_result.loc[mask_filter, cols_round_2] = (
        filt_2_str + " (" + enh_2_str + "\\%)"
    )

    df_result.loc[mask_filter, cols_round_3] = (
        filt_3_str + " (" + enh_3_str + "\\%)"
    )
    
    df_result.loc[mask_filter, ['model_name']] = (
        df_result.loc[mask_filter, ['model_name']] + "$^*$"
    )

    df_result.loc[mask_ori, cols_round_1] = ori_1_str
    df_result.loc[mask_ori, cols_round_2] = ori_2_str
    df_result.loc[mask_ori, cols_round_3] = ori_3_str

df_result = df_result[['model_name', 'mAP', 'NDS', 'GFLOPS', 'peak_memory_allocated_gb', 'fps_1', 'fps_5', 'co2_g']]
cols_name_map = {
    'model_name': 'Model',
    'mAP': 'mAP',
    'NDS': 'NDS',
    'GFLOPS': 'GFLOPS',
    'peak_memory_allocated_gb': 'Mem (GB)',
    'fps_1': 'FPS@1',
    'fps_5': 'FPS@5',
    'co2_g': 'CO2 (g)'
}

df_result = df_result.rename(columns=cols_name_map)

df_result

,Model,mAP,NDS,GFLOPS,Mem (GB),FPS@1,FPS@5,CO2 (g)
0,BEVFusion-L,0.643,0.691,246.09,2.81,21.8,38.6,2.51
1,BEVFusion-L$^*$,0.672 (+4.5\%),0.704 (+1.9\%),185.67 (-24.6\%),2.79 (-0.7\%),26.8 (+22.8\%),61.0 (+57.9\%),1.69 (-32.6\%)
2,BEVFusion-L 3Dh,0.639,0.691,230.48,2.8,17.2,32.9,3.1
3,BEVFusion-L 3Dh$^*$,0.666 (+4.3\%),0.702 (+1.5\%),113.65 (-50.7\%),2.78 (-0.7\%),22.1 (+28.1\%),65.9 (+100.2\%),1.58 (-49.0\%)
4,CenterPoint-Voxel,0.557,0.642,163.53,1.06,14.5,26.4,3.27
5,CenterPoint-Voxel$^*$,0.612 (+10.0\%),0.669 (+4.2\%),121.62 (-25.6\%),0.45 (-57.4\%),17.2 (+18.1\%),39.0 (+47.6\%),1.54 (-52.9\%)
6,CenterPoint-Voxel 3Dh,0.568,0.65,167.3,1.06,5.9,7.9,5.71
7,CenterPoint-Voxel 3Dh$^*$,0.605 (+6.6\%),0.664 (+2.0\%),86.26 (-48.4\%),0.45 (-57.4\%),13.3 (+124.5\%),32.4 (+310.9\%),1.49 (-73.9\%)
8,CenterPoint-Pillar,0.482,0.593,127.86,3.22,22.0,48.9,2.59
9,CenterPoint-Pillar$^*$,0.576 (+19.5\%),0.645 (+8.8\%),127.25 (-0.5\%),0.82 (-74.4\%),34.5 (+56.9\%),71.1 (+45.2\%),1.4 (-46.0\%)


In [21]:
latex_str = '''
\\begin{table*}[htpb]
    \\caption{Performance results of the models. Values in parentheses indicate the percentage improvement over the unfiltered baseline. Models marked with $^*$ are trained and evaluated using filtered point clouds.}
    \\centering
    \\begin{tabular}{lccccccc}
    \\midrule
    Model & mAP & NDS & GFLOPS & Mem (GB) & FPS@1 & FPS@5 & CO$_2$ (g) \\\\
    \\midrule
'''

for index, row in df_result.iterrows():
    latex_str += '    ' + ' & '.join(row.values) + ' \\\\\n'
    
    if index % 2 == 1:
        if not (index == len(df_result) - 1):
            latex_str += '    \\midrule\n'
        else:
            latex_str += '    \\bottomrule\n'

latex_str += '''    \end{tabular}
    \\label{tab:results}
\\end{table*}
'''

with open(os.path.join(plots_and_tables_dir, 'results_table.tex'), 'w') as f:
    f.write(latex_str)

## Generate models plot

In [22]:
import pandas as pd
import numpy as np
import os

In [23]:
df = pd.read_csv(os.path.join(results_root_dir, 'evaluation_results.csv'))
df

,model_name,model_category,trained_on_filter,dataset,filter_type,mAP,NDS,flops,fps_1,fps_5,peak_memory_allocated_gb,peak_memory_reserved_gb,co2_kg
0,bevfusion_lidar_3dh,voxel,False,nuscenes,no_filter,0.638657,0.691101,2.304831e+11,17.216770,32.905030,2.802252,4.031250,0.003098
1,bevfusion_lidar_3dh,voxel,True,nuscenes,1x,0.666361,0.701666,1.136462e+11,22.055646,65.882156,2.782773,3.771484,0.001581
2,bevfusion_lidar_3dh,voxel,False,nuscenes,1x,0.624233,0.679719,1.136452e+11,22.002129,65.935155,2.784235,3.751953,0.001686
3,bevfusion_lidar_3dh,voxel,False,nuscenes,2x,0.622159,0.678945,1.137938e+11,21.987146,65.865696,2.783653,3.783203,0.001597
4,bevfusion_lidar_3dh,voxel,False,nuscenes,4x,0.626127,0.681632,1.144872e+11,21.967718,65.736583,2.783583,3.751953,0.001644
5,bevfusion_lidar_3dh,voxel,False,nuscenes,8x,0.635066,0.686545,1.156407e+11,21.960243,65.069696,2.786489,3.783203,0.001651
6,bevfusion_lidar_3dh,voxel,False,nuscenes,16x,0.639469,0.688805,1.183244e+11,21.808882,64.841081,2.788597,3.751953,0.001674
7,bevfusion_lidar_3dh,voxel,False,nuscenes,32x,0.642577,0.691745,1.226794e+11,21.163964,63.556858,2.788537,3.787109,0.001747
8,bevfusion_lidar,voxel,False,nuscenes,no_filter,0.642859,0.691438,2.460861e+11,21.781513,38.617019,2.805007,4.046875,0.002514
9,bevfusion_lidar,voxel,True,nuscenes,1x,0.671862,0.704306,1.856696e+11,26.757560,60.982850,2.785443,3.802734,0.001694


In [24]:
df_line_plot = df[~((df['filter_type'] == 'no_filter') | ((df['filter_type'] == '1x') & (df['trained_on_filter'] == True)))].copy()

model_names = df['model_name'].unique()

df_line_plot['mAP_decrease'] = 0.0

for model in model_names:
    df_model = df_line_plot[df_line_plot['model_name'] == model]
    base_mAP = df[(df['filter_type'] == 'no_filter') & (df['model_name'] == model) & (~df['trained_on_filter'])]['mAP'].values[0]
    for _, row in df_model.iterrows():
        if row['filter_type'] != 'no_filter':
            df_line_plot.loc[row.name, 'mAP_decrease'] = row['mAP'] - base_mAP

df_line_plot = df_line_plot[['model_name', 'filter_type', 'mAP_decrease', 'model_category']]

models_order = [
    'bevfusion_lidar',
    'bevfusion_lidar_3dh',
    'centerpoint_voxel',
    'centerpoint_voxel_3dh',
    'centerpoint_pillar',
    'ssn',
    'pointpillars'
]
df_line_plot = df_line_plot.set_index('model_name').loc[models_order].reset_index()

models_name_map = {
    'bevfusion_lidar': 'BEVFusion-L',
    'bevfusion_lidar_3dh': 'BEVFusion-L 3Dh',
    'centerpoint_voxel': 'CenterPoint-Voxel',
    'centerpoint_voxel_3dh': 'CenterPoint-Voxel 3Dh',
    'centerpoint_pillar': 'CenterPoint-Pillar',
    'ssn': 'SSN',
    'pointpillars': 'PointPillars'
}
df_line_plot['model_name'] = df_line_plot['model_name'].map(models_name_map)

col_name_map = {
    'model_name': 'Model',
    'filter_type': 'Voxel size',
    'mAP_decrease': 'Delta mAP',
    'model_category': 'Model category'
}
df_line_plot = df_line_plot.rename(columns=col_name_map)

In [ ]:
import plotly.express as px

fig = px.line(
    df_line_plot,
    x='Voxel size',
    y='Delta mAP',
    color='Model',
    symbol='Model category',
    markers=True
)

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    width=315,
    height=250,
    legend=dict(
        title='',
        orientation="h",
        yanchor="top",
        y=1.4,
        xanchor="center",
        x=0.55
    )
)

fig.update_yaxes(
    title=None
)

fig.add_annotation(
    text="Δ mAP",
    x=-0.14,
    y=1.0,
    xref="paper",
    yref="paper",
    showarrow=False,
    xanchor="left",
    yanchor="bottom"
)

fig.update_xaxes(
    title_text="Voxel size",
    title_standoff=4
)

for trace in fig.data:
    trace.name = trace.name.replace(', pillar', '').replace(', voxel', '')

fig.write_image(os.path.join(plots_and_tables_dir, "delta_map_line_plot.png"))
fig.write_image(os.path.join(plots_and_tables_dir, "delta_map_line_plot.svg"))

fig

In [ ]:
with open(os.path.join(plots_and_tables_dir, "delta_map_line_plot.svg"), "r", encoding="utf-8") as f:
    svg = f.read()

svg = svg.replace("−", "-")  # U+2212 → ASCII minus

with open(os.path.join(plots_and_tables_dir, "delta_map_line_plot.svg"), "w", encoding="utf-8") as f:
    f.write(svg)